In [ ]:
input_path = "Input/20241126_Star_Rail/Rail"
file_type = ".TIF"

output_path = "Output"
output_name = "output"

mask_path = "Input/20241126_Star_Rail/Mask/Mask.png"

## Import

In [ ]:
%pip install --upgrade pip
%pip install matplotlib numpy opencv-python tqdm natsort torch torchvision
%pip install ipympl
%matplotlib widget

In [ ]:
import os
import numpy as np
import tifffile as tiff
from tqdm import tqdm
from natsort import natsorted
import matplotlib.pyplot as plt
import cv2

## Tools

In [ ]:
def print_img(img):
    plt.figure(figsize=(10, 6))
    plt.imshow(img)
    plt.show()

In [ ]:
def read_image(path):
    img_ori = tiff.imread(path)

    if img_ori.dtype == np.uint8:
        img_raw = img_ori / 255.0
    elif img_ori.dtype == np.uint16:
        img_raw = img_ori / 65535.0
    elif img_ori.dtype == np.float32 or img_ori.dtype == np.float64:
        pass
        # img_raw = np.clip(ori_img, 0, 1)

    return img_raw

## Setting

In [ ]:
img_paths = [os.path.join(input_path, path) for path in os.listdir(input_path) if path.endswith(file_type)]
img_paths = natsorted(img_paths)

img_shape = tiff.imread(img_paths[0]).shape

## Normal Rail

In [ ]:
def overlay_normal_rail(img_paths, img_shape):
    img_output = np.zeros(img_shape, dtype=np.float32)
    for img in tqdm(img_paths):
        img_raw = read_image(img)
        np.maximum(img_output, img_raw, out=img_output)
    return img_output

In [ ]:
img_normal_rail = overlay_normal_rail(
    img_paths, 
    img_shape
)

print_img(img_normal_rail)

In [ ]:
tiff.imwrite(
    os.path.join(output_path, f"{output_name}_Normal_Rail.tif"), # 修正副檔名寫法
    img_normal_rail.astype(np.float32), 
    photometric='rgb', 
    compression='zlib'
)

## Get The Background

In [ ]:
def overlay_background(img_paths, img_shape):
    output_img = np.zeros(img_shape, dtype=np.float32)
    for img in tqdm(img_paths):
        img_raw = read_image(img)
        np.add(output_img, img_raw, out=output_img)
    return output_img/len(img_paths)

In [ ]:
img_backgorund = overlay_background(
    img_paths,
    img_shape
)

print_img(img_backgorund)

In [ ]:
tiff.imwrite(
    os.path.join(output_path, f"{output_name}_Background.tif"), # 修正副檔名寫法
    img_backgorund.astype(np.float32), 
    photometric='rgb', 
    compression='zlib'
)

## Edit with Mask

In [ ]:
blur_size = 101

mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
mask_float = mask.astype(np.float32) / 255.0 # type: ignore
mask_float = cv2.GaussianBlur(mask_float, (blur_size, blur_size), 0)
mask_3d = mask_float[:, :, np.newaxis]

print_img(mask_3d)

In [ ]:
def generate_scaled_canvas(img, scale, center_x, center_y, target_shape):
    height, width, _ = target_shape
    M = cv2.getRotationMatrix2D((center_x, center_y), 0, scale)
    canvas = cv2.warpAffine(img, M, (width, height), flags=cv2.INTER_LINEAR)
    
    return canvas

def overlay_rotating_rail(img_paths, img_shape, center_x, center_y):
    count = len(img_paths)
    star_trail_sum = np.zeros(img_shape, dtype=np.float32)
    ground_sum = np.zeros(img_shape, dtype=np.float32)

    for ind, picture in tqdm(enumerate(img_paths), total=count):
        n_img = read_image(picture)
        star_img = n_img*mask_3d
        ground_img = n_img*(1-mask_3d)

        current_scale = (ind / (count*1))
        
        height, width, _ = img_shape
        M = cv2.getRotationMatrix2D((center_x, center_y), 0, current_scale)
        canvas = cv2.warpAffine(star_img, M, (width, height), flags=cv2.INTER_LINEAR)

        star_trail_sum = np.maximum(canvas, star_trail_sum)
        ground_sum = np.add(ground_img, ground_sum)

    final_img = star_trail_sum+(0*ground_sum/count)

    return final_img

In [ ]:
img_rotating = overlay_rotating_rail(
    img_paths, img_shape, 
    center_x=1733, # 你想要縮放的中心點 X 座標 (像素)
    center_y=1852  # 你想要縮放的中心點 Y 座標 (像素)
)

print_img(img_rotating)

In [ ]:
tiff.imwrite(
    os.path.join(output_path, f"{output_name}_Rotating_Rail.tif"), # 修正副檔名寫法
    img_rotating.astype(np.float32), 
    photometric='rgb', 
    compression='zlib'
)

# GPU

In [ ]:
import torch
import torchvision.transforms.functional as F
from tqdm import tqdm

def overlay_rotating_rail_gpu_torch_pro(img_paths, img_shape, center_x, center_y, mask_3d, working_scale=4.0, final_scale=2.0):
    count = len(img_paths)
    
    device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
    print(f"Using device: {device} | 運算倍率: {working_scale}x | 最終輸出倍率: {final_scale}x")
    
    # 取得原圖尺寸
    orig_h, orig_w = img_shape[0], img_shape[1]
    
    # 1. 運算用（超高解析度）的尺寸與中心點
    work_h = int(orig_h * working_scale)
    work_w = int(orig_w * working_scale)
    work_center_x = center_x * working_scale
    work_center_y = center_y * working_scale
    
    # 2. 最終輸出的尺寸
    final_h = int(orig_h * final_scale)
    final_w = int(orig_w * final_scale)
    
    # 建立超巨大的畫布
    star_trail_sum = torch.zeros((3, work_h, work_w), dtype=torch.float32, device=device)
    
    mask_tensor = torch.from_numpy(mask_3d).to(torch.float32).permute(2, 0, 1).to(device)
    mask_tensor = F.resize(mask_tensor, [work_h, work_w], F.InterpolationMode.BILINEAR)
    
    for ind, picture in tqdm(enumerate(img_paths), total=count):
        n_img = read_image(picture) 
        img_tensor = torch.from_numpy(n_img).to(torch.float32).permute(2, 0, 1).to(device)
        
        # 每一張圖都先放大到超高解析度
        img_tensor = F.resize(img_tensor, [work_h, work_w], F.InterpolationMode.BILINEAR)
        
        star_img = img_tensor * mask_tensor
        
        current_scale = max((ind / count), 0.001)
            
        # 在超巨大的畫布上進行縮放變形
        canvas = F.affine(
            star_img, 
            angle=0.0, 
            translate=[0, 0], 
            scale=current_scale, 
            shear=0.0, 
            center=[work_center_x, work_center_y],
            interpolation=F.InterpolationMode.BILINEAR
        )
        
        star_trail_sum = torch.maximum(canvas, star_trail_sum)

    # 3. 迴圈結束後，將超巨大畫布「濃縮」回你要的 2 倍大小
    print("正在進行最終抗鋸齒縮放...")
    if working_scale != final_scale:
        final_tensor = F.resize(
            star_trail_sum, 
            [final_h, final_w], 
            F.InterpolationMode.BICUBIC, 
            antialias=True
        )
    else:
        final_tensor = star_trail_sum # 如果 working_scale 跟 final_scale 一樣就直接過
        
    final_img_np = final_tensor.permute(1, 2, 0).cpu().numpy()
    
    return final_img_np

In [ ]:
img_GPU_rotating = overlay_rotating_rail_gpu_torch_pro(
    img_paths, img_shape, 
    center_x=1733,
    center_y=1852,
    mask_3d=mask_3d
)

print_img(img_GPU_rotating)

In [ ]:
tiff.imwrite(
    os.path.join(output_path, f"{output_name}_Rotating_Rail.tif"), # 修正副檔名寫法
    img_GPU_rotating.astype(np.float32), 
    photometric='rgb', 
    compression='zlib'
)